# locations_mapped — nightly rebuild (oxjob #762 / #765)

`locations_mapped` is a nightly `CREATE OR REPLACE`: `locations_w_types` JOIN the
`location_work_ids` identity registry (maintained by the `Map_Work_Ids` task,
oxjob #764), UNION the frozen `locations_stale` sidecar. `location_enrichments`
overlays dormant-writer columns works_base still consumes. `locations_mapped_hash`
recreates the MERGE-era `openalex_updated_dt` semantics: stamps carry when content
is unchanged, bump when it is not. Design + evidence: oxjobs #762 PLAN-v2, #765.

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.location_enrichments') (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  language_classification STRUCT<language: STRING, score: DOUBLE>,
  referenced_works_count INT,
  referenced_works ARRAY<BIGINT>,
  snapshot_dt DATE
)
CLUSTER BY (provenance, native_id_namespace, native_id)

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.locations_stale') (
  work_id BIGINT,
  work_id_source STRING,
  merge_key STRUCT<doi: STRING, pmid: STRING, arxiv: STRING, title_author: STRING>,
  key_lineage STRING,
  provenance STRING,
  native_id STRING,
  native_id_namespace STRING,
  title STRING,
  normalized_title STRING,
  authors ARRAY<STRUCT<given: STRING, family: STRING, name: STRING, orcid: STRING, affiliations: ARRAY<STRUCT<name: STRING, department: STRING, ror_id: STRING>>, is_corresponding: BOOLEAN, author_key: STRING>>,
  ids ARRAY<STRUCT<id: STRING, namespace: STRING, relationship: STRING>>,
  raw_type STRING,
  type STRING,
  version STRING,
  license STRING,
  language STRING,
  language_classification STRUCT<language: STRING, score: DOUBLE>,
  published_date DATE,
  created_date DATE,
  updated_date DATE,
  issue STRING,
  volume STRING,
  first_page STRING,
  last_page STRING,
  is_retracted BOOLEAN,
  abstract STRING,
  source_name STRING,
  publisher STRING,
  funders ARRAY<STRUCT<doi: STRING, ror: STRING, name: STRING, awards: ARRAY<STRING>>>,
  references ARRAY<STRUCT<doi: STRING, pmid: STRING, arxiv: STRING, title: STRING, authors: STRING, year: STRING, raw: STRING>>,
  urls ARRAY<STRUCT<url: STRING, content_type: STRING>>,
  pdf_url STRING,
  landing_page_url STRING,
  pdf_s3_id STRING,
  grobid_s3_id STRING,
  mesh STRING,
  is_oa BOOLEAN,
  is_oa_source BOOLEAN,
  referenced_works_count INT,
  referenced_works ARRAY<BIGINT>,
  abstract_inverted_index STRING,
  authors_exist BOOLEAN,
  affiliations_exist BOOLEAN,
  is_corresponding_exists BOOLEAN,
  best_doi STRING,
  source_id BIGINT,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  violates_landing_rule BOOLEAN,
  snapshot_dt DATE,
  stale_reason STRING
)

## Hash baseline (first run only) — change detection for `openalex_updated_dt`
Payload excludes the two bookkeeping stamps. Formula changes require a rebaseline
from pre-change rows or the next run bumps everything (#733 08-07 incident).

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.locations_mapped_hash')
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       xxhash64(to_json(struct(
         work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
         authors, ids, raw_type, type, version, license, language, language_classification,
         published_date, created_date, updated_date, issue, volume, first_page, last_page,
         is_retracted, abstract, source_name, publisher, funders, references, urls,
         pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
         referenced_works_count, referenced_works, abstract_inverted_index,
         authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
       ))) AS payload_hash,
       openalex_updated_dt
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

## Mojibake repair UDFs (oxjob #801)
Publishers deposit double-encoded UTF-8 ("UniversitÃ©") — ~179K works, crossref-dominant.
The rebuild below repairs affiliation-string mojibake at map time for every provenance;
changed payloads bump `openalex_updated_dt` via the hash mechanism, which propagates the
repair downstream (works_base → work_authors → authorships → ES) with no separate sweep.
Two functions: `repair_mojibake` (scalar; also used by SyncRasCurations to re-key curations)
and `repair_mojibake_names_json` (batch over a JSON array of a row's distinct gate-matching
names, returning only the changed pairs — Spark cannot call an external UDF inside
`transform()` lambdas, and whole-authors-array payloads OOMed the warehouse Python sandbox
on hyperauthorship works, so the rebuild joins the pairs back as a map). Both are conservative by
construction (whole-string strict round-trip + restricted lead signature; 0 false positives
on a 3K clean-string corpus): unrepairable text passes through unchanged. Source of truth +
unit tests: oxjobs #801 `work/mojibake_repair.py`.


In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.repair_mojibake(s STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #801: repairs UTF-8-read-as-latin1/cp1252/latin2/cp1250 mojibake in text (e.g. UniversitÃ© -> Université); returns input unchanged unless confidently repairable. Source of truth + tests: oxjobs #801 work/mojibake_repair.py'
AS $$

import unicodedata

MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep -- each round strictly shrinks

# --- unified latin-1 + cp1252 reverse map ------------------------------------
# chars <= U+00FF map to their own codepoint (latin-1). cp1252's specials
# (which all sit > U+00FF) map to the 0x80-0x9F byte cp1252 decoded them from.
_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

# --- detection ----------------------------------------------------------------
# Core leads: chars whose latin-1 byte is a UTF-8 lead byte for scripts that
# actually occur in affiliations. C3=Ã (Latin-1 sup), C2=Â (Latin-1 punct),
# C4=Ä C5=Å (Latin Ext-A), CE=Î CF=Ï (Greek), D0=Ð D1=Ñ (Cyrillic),
# E2=â (general punctuation, 3-byte).
_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌ\u0088nster")
# Peripheral 2-byte leads: allowed to *participate* in repair but not sufficient
# alone to trigger it (kills the Ö/Ü + NBSP legit-typography corruption risk).
_PERIPHERAL_LEADS_2 = set(
    "ÆÇÒÓÔÕÖØÙÚÛÜ"
)



def _is_cont(ch):
    """Char maps (via unified map) to a UTF-8 continuation byte 0x80-0xBF?"""
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F




def detect(s):
    """True if s contains at least one CORE mojibake sequence (latin-1/cp1252 read)."""
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        # 3-byte leads à-á ã-ï (skip â handled above) need two continuations,
        # at least one of which must not be NBSP: legit "Umeå\xa0\xa0University"
        # (å + typographic NBSPs) is byte-wise valid UTF-8 for a rare CJK char
        # and would otherwise corrupt.
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        # 4-byte leads ð-ô need three continuations (same NBSP rule)
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    """True if s contains a DISTINCTIVE lead+continuation pair under the given
    single-byte codec (ISO-8859-2 / cp1250 reads: UTF-8 lead bytes C3/C4/C5
    decode to Ă/Ä/Ĺ there). "Distinctive" = the lead char maps to a different
    byte than its own codepoint, i.e. only that codec's read produces it —
    without this, latin2's É (0xC9, same as latin-1) would reopen the
    "UNIVERSITÉ\\xa0PARIS" corruption trap the core detector closed."""
    if not s:
        return False
    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None
    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if (b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n):
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


# --- repair -------------------------------------------------------------------

def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)




def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "\ufffd" in fixed:
        return None
    return fixed


def _badness(s):
    """Count of chars implausible in repaired text (symbols, controls). Used to
    pick among codec candidates: a wrong-codec 'repair' that still validates
    leaves stray symbols (e.g. cp1250 text pushed through latin2 yields '÷')."""
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    """One round of repair. Returns repaired string, or None if not repairable.

    All candidate codecs are tried and the least-bad validated result wins —
    ISO-8859-2 and cp1250 reads produce overlapping signatures, and the wrong
    codec can still yield valid-UTF-8 garbage, so first-match is not safe.
    """
    candidates = []
    if detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    # fallback single-byte codecs (Central European reads): the codec's own
    # encoder IS the reverse map -- strict encode fails on any unmappable char.
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    """Map surviving C1 controls (U+0080-U+009F) to their cp1252 chars."""
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair(s):
    """Repair mojibake in s -- returns s unchanged if not confidently repairable."""
    if not s:
        return s
    cur = s
    for _ in range(MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur

return repair(s)
$$

In [ ]:
CREATE OR REPLACE FUNCTION openalex.common.repair_mojibake_names_json(names_json STRING)
RETURNS STRING
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #801: batch-repairs mojibake over a JSON array of affiliation-name strings; returns a JSON array of {o, r} pairs for the strings it changed. Replaces repair_mojibake_authors_json: whole-authors-array payloads OOMed the warehouse Python sandbox on hyperauthorship works, so callers pass only the row''s distinct gate-matching names. Same repair core as openalex.common.repair_mojibake.'
AS $$
import json as _json


import unicodedata

MAX_ROUNDS = 4  # corpus has strings mojibaked up to ~5 levels deep -- each round strictly shrinks

# --- unified latin-1 + cp1252 reverse map ------------------------------------
# chars <= U+00FF map to their own codepoint (latin-1). cp1252's specials
# (which all sit > U+00FF) map to the 0x80-0x9F byte cp1252 decoded them from.
_CP1252_REV = {}
for _b in range(0x80, 0xA0):
    try:
        _CP1252_REV[bytes([_b]).decode("cp1252")] = _b
    except UnicodeDecodeError:
        pass  # 0x81 0x8D 0x8F 0x90 0x9D are undefined in cp1252

# --- detection ----------------------------------------------------------------
# Core leads: chars whose latin-1 byte is a UTF-8 lead byte for scripts that
# actually occur in affiliations. C3=Ã (Latin-1 sup), C2=Â (Latin-1 punct),
# C4=Ä C5=Å (Latin Ext-A), CE=Î CF=Ï (Greek), D0=Ð D1=Ñ (Cyrillic),
# E2=â (general punctuation, 3-byte).
_CORE_LEADS_2 = set("ÃÂÄÅÌÎÏÐÑ")  # Ì (CC) covers NFD combining-mark mojibake ("MuÌ\u0088nster")
# Peripheral 2-byte leads: allowed to *participate* in repair but not sufficient
# alone to trigger it (kills the Ö/Ü + NBSP legit-typography corruption risk).
_PERIPHERAL_LEADS_2 = set(
    "ÆÇÒÓÔÕÖØÙÚÛÜ"
)



def _is_cont(ch):
    """Char maps (via unified map) to a UTF-8 continuation byte 0x80-0xBF?"""
    cp = ord(ch)
    if 0x80 <= cp <= 0xBF:
        return True
    return _CP1252_REV.get(ch, 0) >= 0x80  # cp1252 specials are all 0x80-0x9F




def detect(s):
    """True if s contains at least one CORE mojibake sequence (latin-1/cp1252 read)."""
    if not s:
        return False
    n = len(s)
    for i, ch in enumerate(s):
        if ch in _CORE_LEADS_2 and i + 1 < n and _is_cont(s[i + 1]):
            return True
        if ch == "â" and i + 2 < n and _is_cont(s[i + 1]) and _is_cont(s[i + 2]):
            return True
        cp = ord(ch)
        # 3-byte leads à-á ã-ï (skip â handled above) need two continuations,
        # at least one of which must not be NBSP: legit "Umeå\xa0\xa0University"
        # (å + typographic NBSPs) is byte-wise valid UTF-8 for a rare CJK char
        # and would otherwise corrupt.
        if (0xE0 <= cp <= 0xEF and cp != 0xE2 and i + 2 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2])
                and not (s[i + 1] == "\xa0" and s[i + 2] == "\xa0")):
            return True
        # 4-byte leads ð-ô need three continuations (same NBSP rule)
        if (0xF0 <= cp <= 0xF4 and i + 3 < n
                and _is_cont(s[i + 1]) and _is_cont(s[i + 2]) and _is_cont(s[i + 3])
                and not (s[i + 1] == s[i + 2] == s[i + 3] == "\xa0")):
            return True
    return False


def _fallback_signature(s, codec):
    """True if s contains a DISTINCTIVE lead+continuation pair under the given
    single-byte codec (ISO-8859-2 / cp1250 reads: UTF-8 lead bytes C3/C4/C5
    decode to Ă/Ä/Ĺ there). "Distinctive" = the lead char maps to a different
    byte than its own codepoint, i.e. only that codec's read produces it —
    without this, latin2's É (0xC9, same as latin-1) would reopen the
    "UNIVERSITÉ\\xa0PARIS" corruption trap the core detector closed."""
    if not s:
        return False
    def byte_of(ch):
        try:
            return ch.encode(codec)[0]
        except (UnicodeEncodeError, IndexError):
            return None
    n = len(s)
    for i, ch in enumerate(s):
        b = byte_of(ch)
        if (b is not None and 0xC2 <= b <= 0xDF and b != ord(ch) and i + 1 < n):
            b2 = byte_of(s[i + 1])
            if b2 is not None and 0x80 <= b2 <= 0xBF:
                return True
    return False


# --- repair -------------------------------------------------------------------

def _to_bytes_unified(s):
    out = bytearray()
    for ch in s:
        cp = ord(ch)
        if cp <= 0xFF:
            out.append(cp)
        else:
            b = _CP1252_REV.get(ch)
            if b is None:
                return None
            out.append(b)
    return bytes(out)




def _validate(s, raw):
    if raw is None:
        return None
    try:
        fixed = raw.decode("utf-8", errors="strict")
    except UnicodeDecodeError:
        return None
    if len(fixed) >= len(s) or "\ufffd" in fixed:
        return None
    return fixed


def _badness(s):
    """Count of chars implausible in repaired text (symbols, controls). Used to
    pick among codec candidates: a wrong-codec 'repair' that still validates
    leaves stray symbols (e.g. cp1250 text pushed through latin2 yields '÷')."""
    score = 0
    for ch in s:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            score += 3
        elif cat.startswith("S"):
            score += 1
    return score


def _repair_once(s):
    """One round of repair. Returns repaired string, or None if not repairable.

    All candidate codecs are tried and the least-bad validated result wins —
    ISO-8859-2 and cp1250 reads produce overlapping signatures, and the wrong
    codec can still yield valid-UTF-8 garbage, so first-match is not safe.
    """
    candidates = []
    if detect(s):
        fixed = _validate(s, _to_bytes_unified(s))
        if fixed is not None:
            candidates.append(fixed)
    # fallback single-byte codecs (Central European reads): the codec's own
    # encoder IS the reverse map -- strict encode fails on any unmappable char.
    for codec in ("iso-8859-2", "cp1250"):
        if _fallback_signature(s, codec):
            try:
                raw = s.encode(codec, errors="strict")
            except UnicodeEncodeError:
                continue
            fixed = _validate(s, raw)
            if fixed is not None:
                candidates.append(fixed)
    if not candidates:
        return None
    return min(candidates, key=_badness)  # ties keep candidate order (min is stable)


def _fix_bare_c1(s):
    """Map surviving C1 controls (U+0080-U+009F) to their cp1252 chars."""
    if not any(0x80 <= ord(ch) <= 0x9F for ch in s):
        return s
    out = []
    for ch in s:
        if 0x80 <= ord(ch) <= 0x9F:
            try:
                ch = bytes([ord(ch)]).decode("cp1252")
            except UnicodeDecodeError:
                pass  # 0x81 0x8D 0x8F 0x90 0x9D undefined -> keep
        out.append(ch)
    return "".join(out)


def repair(s):
    """Repair mojibake in s -- returns s unchanged if not confidently repairable."""
    if not s:
        return s
    cur = s
    for _ in range(MAX_ROUNDS):
        nxt = _repair_once(cur)
        if nxt is None:
            break
        cur = nxt
    if cur is not s:
        cur = _fix_bare_c1(cur)
    return cur


if names_json is None:
    return None
try:
    names = _json.loads(names_json)
except Exception:
    return None
out = []
for n in names or []:
    if isinstance(n, str):
        r = repair(n)
        if r != n:
            out.append({"o": n, "r": r})
return _json.dumps(out, ensure_ascii=False)
$$

In [ ]:
DROP FUNCTION IF EXISTS openalex.common.repair_mojibake_authors_json

## Rebuild `locations_mapped`
Live side: `locations_w_types` (one row per anchor) JOIN registry JOIN enrichment
overlay. Stale side: sidecar rows not already represented live — the anti-join is at
`(anchor, work_id)` granularity so preserved twin rows (`stale_reason = 'twin_alive'`)
survive alongside their live sibling; a returning record with the same work_id is
not duplicated. Stamps carry over when the payload hash is unchanged.

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.locations_mapped')
CLUSTER BY (merge_key.doi, merge_key.pmid, merge_key.arxiv, merge_key.title_author)
TBLPROPERTIES (
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
  QUALIFY rwcnt = 1
),
live AS (
  SELECT
    r.work_id,
    r.work_id_source,
    t.merge_key,
    CAST(NULL AS STRING) AS key_lineage,
    t.provenance,
    t.native_id,
    t.native_id_namespace,
    t.title,
    t.normalized_title,
    t.authors,
    t.ids,
    t.raw_type,
    t.type,
    t.version,
    t.license,
    t.language,
    e.language_classification,
    t.published_date,
    t.created_date,
    t.updated_date,
    t.issue,
    t.volume,
    t.first_page,
    t.last_page,
    COALESCE(t.is_retracted, FALSE) AS is_retracted,
    t.abstract,
    t.source_name,
    t.publisher,
    t.funders,
    t.references,
    t.urls,
    t.pdf_url,
    t.landing_page_url,
    t.pdf_s3_id,
    t.grobid_s3_id,
    t.mesh,
    COALESCE(t.is_oa, FALSE) AS is_oa,
    COALESCE(t.is_oa_source, FALSE) AS is_oa_source,
    e.referenced_works_count,
    e.referenced_works,
    t.abstract_inverted_index,
    t.authors_exist,
    t.affiliations_exist,
    t.is_corresponding_exists,
    t.best_doi,
    t.source_id,
    COALESCE(r.openalex_created_dt, current_date()) AS openalex_created_dt
  FROM t
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_enrichments') e
    ON  t.provenance = e.provenance
    AND t.native_id_namespace = e.native_id_namespace
    AND t.native_id = e.native_id
),
stale AS (
  SELECT
    s.work_id,
    s.work_id_source,
    s.merge_key,
    s.key_lineage,
    s.provenance,
    s.native_id,
    s.native_id_namespace,
    s.title,
    s.normalized_title,
    s.authors,
    s.ids,
    s.raw_type,
    s.type,
    s.version,
    s.license,
    s.language,
    s.language_classification,
    s.published_date,
    s.created_date,
    s.updated_date,
    s.issue,
    s.volume,
    s.first_page,
    s.last_page,
    s.is_retracted,
    s.abstract,
    s.source_name,
    s.publisher,
    s.funders,
    s.references,
    s.urls,
    s.pdf_url,
    s.landing_page_url,
    s.pdf_s3_id,
    s.grobid_s3_id,
    s.mesh,
    s.is_oa,
    s.is_oa_source,
    s.referenced_works_count,
    s.referenced_works,
    s.abstract_inverted_index,
    s.authors_exist,
    s.affiliations_exist,
    s.is_corresponding_exists,
    s.best_doi,
    s.source_id,
    s.openalex_created_dt
  FROM identifier('openalex' || :env_suffix || '.works.locations_stale') s
  LEFT ANTI JOIN live l
    ON  s.provenance = l.provenance
    AND s.native_id_namespace = l.native_id_namespace
    AND s.native_id = l.native_id
    AND s.work_id <=> l.work_id
),
unioned AS (
  SELECT * FROM live
  UNION ALL
  SELECT * FROM stale
),
-- oxjob #801: repair mojibake in affiliation strings for ALL provenances at map
-- time. The cheap RLIKE gate (a superset of the UDF's own detector: suspicious
-- lead/continuation pairs incl. cp1252 specials as continuations, plus the
-- latin2/cp1250 signature chars Ă/Ĺ) keeps the Python UDF off ~99.9% of rows.
-- The gate sits in a WHERE and the UDF in a projection above it (Spark hoists
-- UC Python UDFs out of CASE branches — see oxjob #801 PLAN gotchas), and the
-- UDF sees only the row's DISTINCT gate-matching names: the previous
-- whole-authors-array JSON round-trip OOMed the warehouse Python sandbox on
-- hyperauthorship works. Only affiliations[].name changes — author names are
-- deliberately NOT touched (they feed AER matching; oxjob #801).
-- Columns are enumerated to keep the table's column order unchanged.
gate_rows AS (
  SELECT provenance, native_id_namespace, native_id, work_id, authors
  FROM unioned
  WHERE EXISTS(authors, a -> EXISTS(a.affiliations, f -> f.name RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]'))
),
repair_maps AS (
  SELECT
    provenance,
    native_id_namespace,
    native_id,
    work_id,
    map_from_entries(from_json(
      openalex.common.repair_mojibake_names_json(to_json(
        array_distinct(filter(
          flatten(transform(authors, a -> coalesce(transform(a.affiliations, f -> f.name), array()))),
          n -> n RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]')))),
      'ARRAY<STRUCT<o: STRING, r: STRING>>')) AS repair_map
  FROM gate_rows
),
repaired AS (
  SELECT
    u.work_id,
    u.work_id_source,
    u.merge_key,
    u.key_lineage,
    u.provenance,
    u.native_id,
    u.native_id_namespace,
    u.title,
    u.normalized_title,
    CASE
      WHEN m.repair_map IS NOT NULL AND cardinality(m.repair_map) > 0
      THEN transform(u.authors, a -> named_struct(
        'given', a.given,
        'family', a.family,
        'name', a.name,
        'orcid', a.orcid,
        'affiliations', transform(a.affiliations, f -> named_struct(
          'name', COALESCE(try_element_at(m.repair_map, f.name), f.name),
          'department', f.department,
          'ror_id', f.ror_id)),
        'is_corresponding', a.is_corresponding,
        'author_key', a.author_key))
      ELSE u.authors
    END AS authors,
    u.ids,
    u.raw_type,
    u.type,
    u.version,
    u.license,
    u.language,
    u.language_classification,
    u.published_date,
    u.created_date,
    u.updated_date,
    u.issue,
    u.volume,
    u.first_page,
    u.last_page,
    u.is_retracted,
    u.abstract,
    u.source_name,
    u.publisher,
    u.funders,
    u.references,
    u.urls,
    u.pdf_url,
    u.landing_page_url,
    u.pdf_s3_id,
    u.grobid_s3_id,
    u.mesh,
    u.is_oa,
    u.is_oa_source,
    u.referenced_works_count,
    u.referenced_works,
    u.abstract_inverted_index,
    u.authors_exist,
    u.affiliations_exist,
    u.is_corresponding_exists,
    u.best_doi,
    u.source_id,
    u.openalex_created_dt
  FROM unioned u
  LEFT JOIN repair_maps m
    ON  u.provenance = m.provenance
    AND u.native_id_namespace = m.native_id_namespace
    AND u.native_id = m.native_id
    AND u.work_id <=> m.work_id
)
SELECT
  u.* EXCEPT (openalex_created_dt),
  u.openalex_created_dt,
  CASE
    WHEN h.payload_hash = xxhash64(to_json(struct(
      u.work_id, u.work_id_source, u.merge_key, u.key_lineage, u.title, u.normalized_title,
      u.authors, u.ids, u.raw_type, u.type, u.version, u.license, u.language, u.language_classification,
      u.published_date, u.created_date, u.updated_date, u.issue, u.volume, u.first_page, u.last_page,
      u.is_retracted, u.abstract, u.source_name, u.publisher, u.funders, u.references, u.urls,
      u.pdf_url, u.landing_page_url, u.pdf_s3_id, u.grobid_s3_id, u.mesh, u.is_oa, u.is_oa_source,
      u.referenced_works_count, u.referenced_works, u.abstract_inverted_index,
      u.authors_exist, u.affiliations_exist, u.is_corresponding_exists, u.best_doi, u.source_id
    ))) THEN h.openalex_updated_dt
    ELSE current_timestamp()
  END AS openalex_updated_dt
FROM repaired u
LEFT JOIN identifier('openalex' || :env_suffix || '.works.locations_mapped_hash') h
  ON  u.provenance = h.provenance
  AND u.native_id_namespace = h.native_id_namespace
  AND u.native_id = h.native_id
  AND u.work_id <=> h.work_id

## Refresh the hash table from the rebuilt output

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.locations_mapped_hash')
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT provenance, native_id_namespace, native_id, work_id,
       xxhash64(to_json(struct(
         work_id, work_id_source, merge_key, key_lineage, title, normalized_title,
         authors, ids, raw_type, type, version, license, language, language_classification,
         published_date, created_date, updated_date, issue, volume, first_page, last_page,
         is_retracted, abstract, source_name, publisher, funders, references, urls,
         pdf_url, landing_page_url, pdf_s3_id, grobid_s3_id, mesh, is_oa, is_oa_source,
         referenced_works_count, referenced_works, abstract_inverted_index,
         authors_exist, affiliations_exist, is_corresponding_exists, best_doi, source_id
       ))) AS payload_hash,
       openalex_updated_dt
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY provenance, native_id_namespace, native_id, work_id
  ORDER BY openalex_updated_dt DESC NULLS LAST) = 1

In [0]:
SELECT format_number(COUNT(*), 0) AS row_count,
       format_number(COUNT(DISTINCT work_id), 0) AS distinct_works
FROM identifier('openalex' || :env_suffix || '.works.locations_mapped')